# Read First Incoming Gmail
This notebook uses `credentials.json` with the official Google Gmail API client to read your latest incoming email.

In [ ]:
# 1. Install required Google libraries (if not already installed)
%pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
# 2. Authenticate and connect to Gmail API
import os
import base64
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# Read-only permission to read emails
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

creds = None
# token.json stores your access and refresh tokens once authorized
if os.path.exists('token.json'):
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)

# If there are no valid credentials, open browser login window
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    # Save credentials for future runs
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

service = build('gmail', 'v1', credentials=creds)
print("✓ Connected to Gmail API successfully!")

In [ ]:
# 3. Fetch and display the first incoming email
results = service.users().messages().list(userId='me', labelIds=['INBOX'], maxResults=1).execute()
messages = results.get('messages', [])

if not messages:
    print("No emails found in INBOX.")
else:
    msg_id = messages[0]['id']
    msg = service.users().messages().get(userId='me', id=msg_id, format='full').execute()
    payload = msg.get('payload', {})
    headers = {h['name']: h['value'] for h in payload.get('headers', [])}

    print("=" * 50)
    print(f"From:    {headers.get('From', 'Unknown')}")
    print(f"Subject: {headers.get('Subject', '(No Subject)')}")
    print(f"Date:    {headers.get('Date', 'Unknown')}")
    print(f"Snippet: {msg.get('snippet', '')}")
    print("=" * 50)

    # Helper function to extract plain text body
    def get_text_body(payload):
        if 'parts' in payload:
            for part in payload['parts']:
                if part.get('mimeType') == 'text/plain' and 'data' in part.get('body', {}):
                    return base64.urlsafe_b64decode(part['body']['data']).decode('utf-8', errors='ignore')
                elif 'parts' in part:
                    text = get_text_body(part)
                    if text:
                        return text
        elif 'body' in payload and 'data' in payload['body']:
            return base64.urlsafe_b64decode(payload['body']['data']).decode('utf-8', errors='ignore')
        return ""

    body = get_text_body(payload)
    if body:
        print("\n--- Body Content ---\n")
        print(body.strip())
    else:
        print("\n(No plain text body content found)")